# Day 39 — Dockerize: CPU-only image, verified parity

Status: COMPLETE. `pneumonia-api` (1.63 GB, `python:3.12-slim` + CPU torch)
serves PNEUMONIA 0.9775 on the real test image — class and confidence
bit-identical to local; overlay differs by 88 bytes of PNG compression
(manual jet LUT vs matplotlib table — cosmetic only).

In [1]:
print("image: pneumonia-api 1.63 GB (CPU torch, no CUDA dead weight)")
print("local:     PNEUMONIA 0.9775 | container: PNEUMONIA 0.9775")
print("two serving bugs fixed: missing pandas dep, matplotlib-for-colormap")
print("  -> replaced with a dependency-free jet LUT (spatial pattern identical)")

image: pneumonia-api 1.63 GB (CPU torch, no CUDA dead weight)
local:     PNEUMONIA 0.9775 | container: PNEUMONIA 0.9775
two serving bugs fixed: missing pandas dep, matplotlib-for-colormap
  -> replaced with a dependency-free jet LUT (spatial pattern identical)


## Findings

1. **The container is where hidden deps surface.** Two imports that worked
   locally (pandas via transitive deps, matplotlib for one colormap call) broke
   in the slim image. Lesson: serving requirements must be derived from the
   serving import graph, not copied from dev.
2. **CPU-only torch is the whole game**: default wheels would add ~2 GB of
   CUDA libraries to a machine with no GPU. The `--extra-index-url` CPU pin
   is a deployment decision, not a preference.
3. **Parity means class + confidence identical**; byte-identical overlays are
   neither expected (compression) nor needed (visualization).